# RUDRA - Physics-Informed Turbomachine Prognostics
### Component Health Indices & Remaining Useful Life (NASA N-CMAPSS DS01 Benchmark)

---
### Dataset Scope & Architecture Notes:
- **Benchmark Dataset**: NASA N-CMAPSS `N-CMAPSS_DS01-005.h5` (~1.2 GB, `bishals098/nasa-cmapss-2-engine-degradation`).
- **Physical Fault Mode**: Per NASA documentation (Arias Chao et al., 2021), **DS01 specifically injects High-Pressure Turbine (HPT) degradation** (flow capacity and efficiency modifiers in `T_dev`). Fan and LPC/HPC stages remain nominal in this subset.
- **True Station Thermodynamics**: Uses exact aerothermal station definitions (Station 2 = Fan inlet, 24 = LPC exit, 30 = HPC exit, 40 = Combustor exit, 48 = HPT exit, 50 = LPT exit).
- **Zero Target Leakage**: `cycle` is completely removed from the feature set. RUL is predicted strictly from thermodynamic physics, cumulative stress integrals, and inferred component health.
- **Inductive Regime Features**: Altitude/Mach regime bins and median baselines are fit strictly on training engines and applied to test engines.


In [ ]:
# -- Cell 1: Package Installation & Serializable Estimators ----------------
import subprocess, sys

def pip(*pkgs): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for mod, pkg in [('lightgbm','lightgbm'), ('xgboost','xgboost'), ('catboost','catboost'),
                  ('shap','shap'), ('h5py','h5py'), ('kaggle','kaggle')]:
    try: __import__(mod)
    except ImportError: pip(pkg)

import lightgbm as lgb
import catboost as cb; from catboost import CatBoostRegressor
import xgboost as xgb
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np, pandas as pd, h5py, os, json, joblib, warnings, shutil, time
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)

# Module-level serializable class for un-degraded components (prevents joblib pickling error)
class ZeroWearIdentityMeta:
    def __init__(self, val=0.0, n_models=4):
        self.val = float(val)
        self.coef_ = np.zeros(n_models, dtype=float)
        self.intercept_ = float(val)
    def predict(self, X):
        return np.full(len(X), self.val, dtype=float)

class ZeroWearIdentityModel:
    def __init__(self, val=0.0):
        self.val = float(val)
    def fit(self, X, y): return self
    def predict(self, X): return np.full(len(X), self.val, dtype=float)
    def save_model(self, path, *args, **kwargs):
        with open(path, 'w') as f: json.dump({'constant': self.val}, f)

print('Environment Ready:')
print(f'   LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | CatBoost {cb.__version__} | Sklearn {sklearn.__version__}')


In [ ]:
# -- Cell 2: Kaggle Authentication & Targeted H5 Download ------------------
import os, shutil
from google.colab import files

KAGGLE_JSON_PATH = os.path.expanduser('~/.kaggle/kaggle.json')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

if not os.path.exists(KAGGLE_JSON_PATH):
    print('Please upload your fresh kaggle.json file:')
    uploaded = files.upload()
    for fn, data in uploaded.items():
        with open(KAGGLE_JSON_PATH, 'wb') as f: f.write(data)
    os.chmod(KAGGLE_JSON_PATH, 0o600)
    print('kaggle.json saved successfully.')
else:
    print('kaggle.json already present.')

DATASET_SLUG = 'bishals098/nasa-cmapss-2-engine-degradation'
TARGET_FILE  = 'N-CMAPSS_DS01-005.h5'
DATA_DIR     = '/content/ncmapss'
H5_PATH      = os.path.join(DATA_DIR, TARGET_FILE)
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(H5_PATH):
    print(f'Downloading {TARGET_FILE} (~1.2 GB)...')
    !kaggle datasets download -d {DATASET_SLUG} -f {TARGET_FILE} -p {DATA_DIR} --unzip
    print('Download complete.')
else:
    print(f'{TARGET_FILE} already present ({os.path.getsize(H5_PATH)/1e9:.2f} GB)')
print(f'H5 path: {H5_PATH}')


In [ ]:
# -- Cell 3: Inspect HDF5 Keys & Health Parameters -------------------------
with h5py.File(H5_PATH, 'r') as hf:
    print('Available HDF5 Keys:')
    for k in sorted(hf.keys()):
        print(f'  {k:12s} shape={hf[k].shape} dtype={hf[k].dtype}')
    if 'T_dev' in hf:
        print(f'\nT_dev shape: {hf["T_dev"].shape}')
        print('  T_dev contains NASA component modifiers (fan, LPC, HPC, HPT, LPT)')


In [ ]:
# -- Cell 4: Ingestion & per-Cycle Aggregation ----------------------------
XS_COLS = ['T24','T30','T48','T50','P15','P2','P21','P24','Ps30','P40','P50','Nf','Nc','Wf']
W_COLS  = ['alt','Mach','TRA','T2']
T_DEV_COLS = [
    'fan_eff_mod','fan_flow_mod',
    'LPC_eff_mod','LPC_flow_mod',
    'HPC_eff_mod','HPC_flow_mod',
    'HPT_eff_mod','HPT_flow_mod',
    'LPT_eff_mod','LPT_flow_mod',
]

CHUNK_SIZE = 500_000

def load_and_downsample(h5_path, subset='dev'):
    W_key  = f'W_{subset}'; Xs_key = f'X_s_{subset}'
    Td_key = f'T_{subset}'; Y_key  = f'Y_{subset}'; A_key  = f'A_{subset}'

    with h5py.File(h5_path, 'r') as hf:
        available = list(hf.keys())
        if Xs_key not in available:
            subset = 'dev'; Xs_key = 'X_s_dev'; W_key = 'W_dev'
            Td_key = 'T_dev'; Y_key = 'Y_dev'; A_key = 'A_dev'
        N = hf[A_key].shape[0]
        print(f'Ingesting {N:,} 1-Hz rows from {subset}...')

        chunks_W, chunks_Xs, chunks_Td, chunks_Y, chunks_A = [], [], [], [], []
        for start in range(0, N, CHUNK_SIZE):
            end = min(start + CHUNK_SIZE, N)
            chunks_W.append(hf[W_key][start:end])
            chunks_Xs.append(hf[Xs_key][start:end])
            td_chunk = hf[Td_key][start:end]
            chunks_Td.append(td_chunk[:, :min(td_chunk.shape[1], len(T_DEV_COLS))])
            chunks_Y.append(hf[Y_key][start:end])
            chunks_A.append(hf[A_key][start:end])

    W  = np.concatenate(chunks_W,  axis=0)
    Xs = np.concatenate(chunks_Xs, axis=0)
    Td = np.concatenate(chunks_Td, axis=0)
    Y  = np.concatenate(chunks_Y,  axis=0).ravel()
    A  = np.concatenate(chunks_A,  axis=0)

    n_tcols = Td.shape[1]
    used_td_cols = T_DEV_COLS[:n_tcols]

    df = pd.DataFrame(
        np.hstack([A[:,:4], W, Xs, Td, Y.reshape(-1,1)]),
        columns=['unit','cycle','flight_class','hs'] + W_COLS + XS_COLS + used_td_cols + ['RUL']
    )
    df['unit']  = df['unit'].astype(int)
    df['cycle'] = df['cycle'].astype(int)
    df['hs']    = df['hs'].astype(int)
    df['RUL']   = df['RUL'].astype(float)

    agg_df = df.groupby(['unit','cycle']).median().reset_index()
    agg_df['unit']  = agg_df['unit'].astype(int)
    agg_df['cycle'] = agg_df['cycle'].astype(int)
    agg_df['hs']    = agg_df['hs'].round().astype(int)
    agg_df = agg_df.sort_values(['unit','cycle']).reset_index(drop=True)
    print(f'Aggregated to {len(agg_df):,} flight cycles across {agg_df["unit"].nunique()} engines.')
    return agg_df, used_td_cols

df_dev, USED_TD_COLS = load_and_downsample(H5_PATH, subset='dev')
print(df_dev[['unit','cycle','alt','Mach','T24','hs','RUL']].head(3))


In [ ]:
# -- Cell 5: Engine Split & Ground Truth Health Target Normalization --------
all_units = sorted(df_dev['unit'].unique())
n_train   = max(int(0.70 * len(all_units)), 1)
train_units = all_units[:n_train]
test_units  = all_units[n_train:]

train_df = df_dev[df_dev['unit'].isin(train_units)].reset_index(drop=True)
test_df  = df_dev[df_dev['unit'].isin(test_units)].reset_index(drop=True)

print(f'Train Engines: {train_units} ({len(train_df)} cycles)')
print(f'Test Engines:  {test_units} ({len(test_df)} cycles)')

COMP_TARGET_MAP = {
    'HI_fan'        : ['fan_eff_mod','fan_flow_mod'],
    'HI_compressor' : ['LPC_eff_mod','LPC_flow_mod','HPC_eff_mod','HPC_flow_mod'],
    'HI_turbine'    : ['HPT_eff_mod','HPT_flow_mod','LPT_eff_mod','LPT_flow_mod'],
}
COMP_TARGET_MAP = {k: [c for c in v if c in USED_TD_COLS] for k, v in COMP_TARGET_MAP.items()}
COMP_TARGET_MAP = {k: v for k, v in COMP_TARGET_MAP.items() if v}

def compute_hi_from_modifiers(df, comp_cols, max_deg_dict=None, is_train=True):
    result = pd.DataFrame(index=df.index)
    new_max = {}
    for comp, cols in comp_cols.items():
        present = [c for c in cols if c in df.columns]
        if not present: continue
        
        baseline = df.groupby('unit')[present].transform(lambda s: s.iloc[:min(5, len(s))].mean())
        wear = (df[present] - baseline).abs().mean(axis=1)
        
        if is_train:
            max_w = float(np.percentile(wear, 99))
            max_w = max(max_w, 1e-4)
            new_max[comp] = max_w
        else:
            max_w = max_deg_dict.get(comp, 0.05)
            
        result[comp] = (wear / max_w).clip(0.0, 1.0)
    return result, new_max

hi_train, max_deg_dict = compute_hi_from_modifiers(train_df, COMP_TARGET_MAP, is_train=True)
hi_test, _             = compute_hi_from_modifiers(test_df,  COMP_TARGET_MAP, max_deg_dict, is_train=False)

for col in hi_train.columns:
    train_df[col] = hi_train[col].values
    test_df[col]  = hi_test[col].values

HI_TARGETS = list(hi_train.columns)
print(f'Component Health Targets: {HI_TARGETS}')


In [ ]:
# -- Cell 6: Aerothermal Station Thermodynamics & Inductive Regime Bins -----
GAMMA_AIR = 1.40; GAMMA_GAS = 1.33

def compute_station_thermodynamics(df):
    d = df.copy()
    
    # Correct Station Thermodynamics per NASA N-CMAPSS definitions:
    # Station 2: Fan inlet | Station 24: LPC exit | Station 30: HPC exit
    # Station 40: Combustor exit | Station 48: HPT exit | Station 50: LPT exit
    d['PR_LPC'] = (d['P24'] / d['P2'].replace(0, np.nan)).clip(1.0, 5.0).fillna(1.8)
    d['PR_HPC'] = (d['Ps30'] / d['P24'].replace(0, np.nan)).clip(1.0, 30.0).fillna(12.0)
    d['PR_total'] = (d['Ps30'] / d['P2'].replace(0, np.nan)).clip(1.0, 50.0).fillna(22.0)
    
    # HPC Isentropic Ideal Exit Temperature (Inlet is Station 24)
    exp_c = (GAMMA_AIR - 1) / GAMMA_AIR
    d['T30_ideal_HPC'] = d['T24'] * (d['PR_HPC']) ** exp_c
    d['eta_HPC_est']   = ((d['T30_ideal_HPC'] - d['T24']) / (d['T30'] - d['T24']).replace(0, np.nan)).clip(0.5, 1.2).fillna(0.85)
    d['res_T30_HPC']   = d['T30'] - d['T30_ideal_HPC']
    
    # Combustor pressure ratio and LPT expansion ratio
    d['PR_combustor'] = (d['P40'] / d['Ps30'].replace(0, np.nan)).clip(0.85, 1.05).fillna(0.96)
    d['ER_LPT']       = (d['P50'] / d['P40'].replace(0, np.nan)).clip(0.01, 1.0).fillna(0.4)
    
    # Turbine temperature margin drift
    d['SFC_proxy'] = d['Wf'] / (d['Nf'] * d['Nc']).replace(0, np.nan)
    d['SFC_proxy'] = d['SFC_proxy'].fillna(d['SFC_proxy'].median())
    
    # Cumulative thermal and mechanical stress integrals
    d['cum_thermal_wear']  = d.groupby('unit')['T48'].cumsum()
    d['cum_pressure_load'] = d.groupby('unit')['PR_HPC'].cumsum()
    d['cum_fuel_burn']     = d.groupby('unit')['Wf'].cumsum()
    
    # Multi-window rolling velocities (3, 7, 15 spans)
    for span in [3, 7, 15]:
        d[f'T48_ma_{span}']     = d.groupby('unit')['T48'].transform(lambda s: s.rolling(span, min_periods=1).mean())
        d[f'T48_slope_{span}']  = d.groupby('unit')['T48'].transform(lambda s: s.diff(span).fillna(0))
        d[f'PR_HPC_ma_{span}']  = d.groupby('unit')['PR_HPC'].transform(lambda s: s.rolling(span, min_periods=1).mean())
    
    return d

train_feat = compute_station_thermodynamics(train_df)
test_feat  = compute_station_thermodynamics(test_df)

# Strictly Inductive Regime Bins: Fit quantile boundaries strictly on train
alt_q  = np.percentile(train_feat['alt'].dropna(), [0, 25, 50, 75, 100])
mach_q = np.percentile(train_feat['Mach'].dropna(), [0, 25, 50, 75, 100])
alt_q[0] -= 1e-3; alt_q[-1] += 1e-3
mach_q[0] -= 1e-3; mach_q[-1] += 1e-3

train_feat['alt_b'] = pd.cut(train_feat['alt'], bins=alt_q, labels=False)
train_feat['mac_b'] = pd.cut(train_feat['Mach'], bins=mach_q, labels=False)
train_feat['regime_id'] = train_feat['alt_b'].astype(str) + '_' + train_feat['mac_b'].astype(str)

# Compute regime expected baseline on train only
regime_map = train_feat.groupby('regime_id')['T48'].median().to_dict()
global_t48 = float(train_feat['T48'].median())
train_feat['T48_regime_residual'] = train_feat['T48'] - train_feat['regime_id'].map(regime_map).fillna(global_t48)

# Apply strictly inductive mapping to test
test_feat['alt_b'] = pd.cut(test_feat['alt'].clip(alt_q[0], alt_q[-1]), bins=alt_q, labels=False)
test_feat['mac_b'] = pd.cut(test_feat['Mach'].clip(mach_q[0], mach_q[-1]), bins=mach_q, labels=False)
test_feat['regime_id'] = test_feat['alt_b'].astype(str) + '_' + test_feat['mac_b'].astype(str)
test_feat['T48_regime_residual'] = test_feat['T48'] - test_feat['regime_id'].map(regime_map).fillna(global_t48)

# Z-score normalization against healthy baseline (first 10 cycles)
BASE_K = 10
SENSOR_COLS = XS_COLS + W_COLS
healthy_mask = train_feat['cycle'] <= BASE_K
ref = train_feat[healthy_mask][SENSOR_COLS].agg(['mean','std']).replace(0, np.nan).fillna(1.0)

for col in SENSOR_COLS:
    for df_f in [train_feat, test_feat]:
        df_f[col+'_z'] = (df_f[col] - ref.loc['mean', col]) / ref.loc['std', col]

ZSCORE_COLS = [c+'_z' for c in SENSOR_COLS]
print('Thermodynamic modeling and inductive regime normalization complete.')


In [ ]:
# -- Cell 7: Feature Set Assembly (Zero 'cycle' shortcut) ------------------
# NOTE: 'cycle' is intentionally EXCLUDED from all feature sets.
# Models must predict RUL from physical degradation, not the cycle counter.

PHYSICS_FEATS = [
    'PR_LPC','PR_HPC','PR_total','PR_combustor','ER_LPT',
    'T30_ideal_HPC','eta_HPC_est','res_T30_HPC','T48_regime_residual','SFC_proxy',
    'cum_thermal_wear','cum_pressure_load','cum_fuel_burn',
    'T48_ma_3','T48_slope_3','PR_HPC_ma_3',
    'T48_ma_7','T48_slope_7','PR_HPC_ma_7',
    'T48_ma_15','T48_slope_15',
] + ZSCORE_COLS

PHYSICS_FEATS = [f for f in PHYSICS_FEATS if f in train_feat.columns and f in test_feat.columns]
train_feat[PHYSICS_FEATS] = train_feat[PHYSICS_FEATS].fillna(0.0)
test_feat[PHYSICS_FEATS]  = test_feat[PHYSICS_FEATS].fillna(0.0)
print(f'Active Physics & Stress Features: {len(PHYSICS_FEATS)} (zero cycle leakage)')


In [ ]:
# -- Cell 8: Layer 4 Training - Stage A (Components) & Stage B (HI -> RUL) --
ALGOS = ['lgb','cat','xgb','rf']
OUT_DIR = '/content/rudra_ncmapss_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'native_models'), exist_ok=True)

def get_base_models(seed=42):
    return {
        'lgb': lgb.LGBMRegressor(n_estimators=250, learning_rate=0.04, num_leaves=24,
                                 min_child_samples=15, subsample=0.85, subsample_freq=1,
                                 colsample_bytree=0.6, reg_lambda=4.0, verbose=-1,
                                 n_jobs=-1, random_state=seed),
        'cat': CatBoostRegressor(iterations=250, learning_rate=0.045, depth=5,
                                 l2_leaf_reg=4, verbose=0, random_seed=seed),
        'xgb': xgb.XGBRegressor(n_estimators=200, learning_rate=0.045, max_depth=5,
                                min_child_weight=4, subsample=0.85, colsample_bytree=0.6,
                                reg_lambda=4.0, tree_method='hist', n_jobs=-1, random_state=seed),
        'rf':  RandomForestRegressor(n_estimators=80, max_depth=12, max_features=0.5,
                                     n_jobs=-1, random_state=seed)
    }

def train_stacked(X_train_df, y_train, X_test_df, feat_cols, groups, target_name):
    t0 = time.time()
    
    # Zero-variance handling for un-degraded stages in DS01
    if np.std(y_train) < 1e-5:
        const_val = float(np.mean(y_train))
        print(f'  [{target_name:20s}] No fault injected in DS01 (constant {const_val:.2f}).')
        dummy_meta = ZeroWearIdentityMeta(const_val, len(ALGOS))
        dummy_prod = {a: ZeroWearIdentityModel(const_val) for a in ALGOS}
        return {'target': target_name, 'meta': dummy_meta, 'prod': dummy_prod,
                'oof_preds': np.full(len(X_train_df), const_val),
                'test_preds': np.full(len(X_test_df), const_val), 'oof_rmse': 0.0, 'has_variance': False}

    n_splits = min(len(np.unique(groups)), 4)
    gkf = GroupKFold(n_splits=n_splits)
    oof = {a: np.zeros(len(X_train_df)) for a in ALGOS}

    Xtr = np.ascontiguousarray(X_train_df[feat_cols].values, dtype=np.float32)
    ytr = np.ascontiguousarray(y_train, dtype=np.float32)
    Xte = np.ascontiguousarray(X_test_df[feat_cols].values, dtype=np.float32)

    for fold, (tr_i, va_i) in enumerate(gkf.split(X_train_df, y_train, groups)):
        mdls = get_base_models(seed=42 + fold)
        for a in ALGOS:
            mdls[a].fit(Xtr[tr_i], ytr[tr_i])
            oof[a][va_i] = mdls[a].predict(Xtr[va_i])

    OOF = np.column_stack([oof[a] for a in ALGOS])
    meta = Ridge(alpha=5.0, fit_intercept=True, positive=True)
    meta.fit(OOF, ytr)

    # Fit final production models on 100% training data
    prod = get_base_models(seed=42)
    for a in ALGOS: prod[a].fit(Xtr, ytr)

    # Generate test predictions using the exact production models
    prod_test_preds = np.column_stack([prod[a].predict(Xte) for a in ALGOS])
    test_pred = meta.predict(prod_test_preds)
    oof_pred  = meta.predict(OOF)
    oof_rmse  = float(np.sqrt(mean_squared_error(ytr, oof_pred)))
    oof_r2    = float(r2_score(ytr, oof_pred))
    print(f'  [{target_name:20s}] OOF-R2={oof_r2:.4f} | OOF-RMSE={oof_rmse:.4f} ({time.time()-t0:.1f}s)')
    return {'target': target_name, 'meta': meta, 'prod': prod,
            'oof_preds': oof_pred, 'test_preds': test_pred, 'oof_rmse': oof_rmse, 'oof_r2': oof_r2, 'has_variance': True}

groups = train_feat['unit'].values
print('=== STAGE A: Training Component Health Models ===')
results = {}
for target in HI_TARGETS:
    results[target] = train_stacked(train_feat, train_feat[target].values,
                                    test_feat,  PHYSICS_FEATS, groups, target)
    train_feat[f'pred_{target}'] = np.clip(results[target]['oof_preds'], 0.0, 1.0)
    test_feat[f'pred_{target}']  = np.clip(results[target]['test_preds'], 0.0, 1.0)

comp_pred_cols = [f'pred_{t}' for t in HI_TARGETS]
train_feat['pred_HI_overall'] = train_feat[comp_pred_cols].mean(axis=1)
test_feat['pred_HI_overall']  = test_feat[comp_pred_cols].mean(axis=1)

# Stage B: Predict RUL conditioned on physics features + predicted component health
for df_b in [train_feat, test_feat]:
    df_b['HI_squared'] = df_b['pred_HI_overall'] ** 2
    df_b['HI_sqrt']    = np.sqrt(df_b['pred_HI_overall'].clip(0, 1))

print('\n=== STAGE B: Training Physics-Informed HI -> RUL Stack ===')
RUL_FEATS = PHYSICS_FEATS + comp_pred_cols + ['pred_HI_overall', 'HI_squared', 'HI_sqrt']
results['RUL'] = train_stacked(train_feat, train_feat['RUL'].values,
                               test_feat,  RUL_FEATS, groups, 'RUL')
print('\nModel Training Complete.')


In [ ]:
# -- Cell 9: Adaptive Kalman Filter & Component-by-Component Evaluation -----
# Adaptive Kalman Filter: Prior drift rate adapts to the ML model's measurement velocity
class AdaptiveRULKalman:
    def __init__(self, q_rul=0.5, q_rate=5e-3, r=25.0):
        self.q_rul, self.q_rate, self.r = q_rul, q_rate, r
        self.x = None
    def step(self, z, dt=1.0):
        if self.x is None:
            self.x = np.array([z, -1.0])
            self.P = np.diag([self.r, 1.0])
            self.last_z = z
            return self.x[0]
        F = np.array([[1.0, dt], [0.0, 1.0]])
        self.x = F @ self.x
        Q = np.array([[self.q_rul, 0.0], [0.0, self.q_rate]])
        self.P = F @ self.P @ F.T + Q
        # Adaptive rate tracking from empirical measurement gradient
        meas_rate = np.clip((z - self.last_z) / dt, -3.0, 0.0)
        self.last_z = z
        self.x[1] = 0.70 * self.x[1] + 0.30 * meas_rate
        S = self.P[0, 0] + self.r
        K = self.P[:, 0] / S
        self.x = self.x + K * (z - self.x[0])
        self.P = self.P - np.outer(K, self.P[0, :])
        return max(self.x[0], 0.0)

def run_adaptive_kalman(df, raw_preds):
    smoothed = np.zeros(len(raw_preds))
    for u in df['unit'].unique():
        idx = np.where(df['unit'].values == u)[0]
        kf = AdaptiveRULKalman()
        for i in idx: smoothed[i] = kf.step(raw_preds[i])
    return smoothed

raw_test_rul    = np.clip(results['RUL']['test_preds'], 0.0, 1000.0)
kalman_test_rul = run_adaptive_kalman(test_feat, raw_test_rul)
results['RUL']['kalman_test_preds'] = kalman_test_rul

true_rul_all   = test_feat['RUL'].values
raw_r2_all     = float(r2_score(true_rul_all, raw_test_rul))
raw_rmse_all   = float(np.sqrt(mean_squared_error(true_rul_all, raw_test_rul)))
raw_mae_all    = float(mean_absolute_error(true_rul_all, raw_test_rul))
kalman_r2_all  = float(r2_score(true_rul_all, kalman_test_rul))
kalman_rmse_all= float(np.sqrt(mean_squared_error(true_rul_all, kalman_test_rul)))
kalman_mae_all = float(mean_absolute_error(true_rul_all, kalman_test_rul))

# Honest Component Metrics: Distinguish between active degrading targets and unmodeled zero-wear targets
comp_metrics = {}
for tgt in HI_TARGETS:
    gt = test_feat[tgt].values
    pr = np.clip(results[tgt]['test_preds'], 0.0, 1.0)
    if np.std(gt) < 1e-5:
        comp_metrics[tgt] = {'r2_str': 'N/A (Var=0)', 'rmse': 0.0, 'mae': 0.0, 'note': 'Zero wear in DS01 benchmark'}
    else:
        c_r2   = float(r2_score(gt, pr))
        c_rmse = float(np.sqrt(mean_squared_error(gt, pr)))
        c_mae  = float(mean_absolute_error(gt, pr))
        comp_metrics[tgt] = {'r2_str': f'{c_r2:.4f}', 'rmse': c_rmse, 'mae': c_mae, 'note': 'Active Fault Degradation'}

print('='*86)
print('             COMPONENT-WISE & SYSTEM PROGNOSTICS ACCURACY (R^2, RMSE, MAE)')
print('='*86)
print(f'  {"Component / Target":28s} | {"R^2 Score":12s} | {"RMSE":11s} | {"MAE":10s} | {"Evaluation Note":20s}')
print('-'*86)
for tgt, m in comp_metrics.items():
    name = tgt.replace('HI_','').capitalize() + ' Health Index'
    print(f'  {name:28s} | {m["r2_str"]:12s} | {m["rmse"]:10.4f}  | {m["mae"]:9.4f}  | {m["note"]:20s}')
print('-'*86)
print(f'  {"Layer 4: Raw Ensemble RUL":28s} | {raw_r2_all:12.4f} | {raw_rmse_all:7.2f} cyc | {raw_mae_all:6.2f} cyc | ML Stack (Zero Cycle Leakage)')
print(f'  {"Layer 5: Adaptive Kalman RUL":28s} | {kalman_r2_all:12.4f} | {kalman_rmse_all:7.2f} cyc | {kalman_mae_all:6.2f} cyc | Dynamic Rate Tracking')
print('='*86)
print(f'\n>>> [SUMMARY] FINAL SYSTEM RUL RMSE: {kalman_rmse_all:.2f} FLIGHT CYCLES | R^2: {kalman_r2_all:.4f} <<<\n')


In [ ]:
# -- Cell 10: Three-Panel Diagnostics (with NASA Ground-Truth 'hs' Flag) -----
fig, ax = plt.subplots(1, 3, figsize=(18, 5.2), dpi=150)

true_rul = test_feat['RUL'].values
pred_rul = raw_test_rul
err      = pred_rul - true_rul

# Panel 1: Parity Plot
max_val = max(true_rul.max(), pred_rul.max()) + 5
ax[0].scatter(pred_rul, true_rul, s=8, alpha=0.5, color='#2980b9', edgecolors='none')
ax[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='1:1 Parity')
ax[0].set_title(f'Parity (R2={raw_r2_all:.3f})', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Predicted RUL', fontsize=10)
ax[0].set_ylabel('Actual RUL', fontsize=10)
ax[0].set_xlim(-2, max_val); ax[0].set_ylim(-2, max_val)
ax[0].grid(True, alpha=0.3)
ax[0].legend(loc='lower right')

# Panel 2: Sorted by True RUL
sort_idx = np.argsort(true_rul)
ax[1].scatter(np.arange(len(true_rul)), pred_rul[sort_idx], s=5, alpha=0.45, color='#2980b9', label='Predictions')
ax[1].plot(np.arange(len(true_rul)), true_rul[sort_idx], 'r-', lw=2.2, label='True RUL')
ax[1].set_title('Sorted by true RUL', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Sorted Sample Index', fontsize=10)
ax[1].set_ylabel('RUL', fontsize=10)
ax[1].grid(True, alpha=0.3)
ax[1].legend(loc='upper left')

# Panel 3: Error by Official NASA Health State Flag (A_dev 'hs')
hs_mask = (test_feat['hs'] == 1).values
box_data = [err[hs_mask], err[~hs_mask]]
bp = ax[2].boxplot(box_data, patch_artist=True, labels=['Healthy (hs=1)', 'Degraded (hs=0)'])
colors = ['#d5f5e3', '#fadbd8']
for patch, color in zip(bp['boxes'], colors): patch.set_facecolor(color)
ax[2].set_title('Error by health state', fontsize=12, fontweight='bold')
ax[2].set_ylabel('Error (Predicted - True RUL)', fontsize=10)
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'three_panel_evaluation.png'), dpi=180)
plt.show()

print('\nWorst test engines by RMSE (raw stack):')
engine_eval = []
for u in test_feat['unit'].unique():
    u_mask = test_feat['unit'] == u
    u_true = true_rul[u_mask]
    u_pred = pred_rul[u_mask]
    u_rmse = float(np.sqrt(mean_squared_error(u_true, u_pred)))
    u_mae  = float(mean_absolute_error(u_true, u_pred))
    u_r2   = float(r2_score(u_true, u_pred)) if np.std(u_true) > 0 else 0.0
    engine_eval.append({'Unit': int(u), 'RMSE': u_rmse, 'MAE': u_mae, 'R2': u_r2, 'Cycles': len(u_true)})

print(pd.DataFrame(engine_eval).sort_values('RMSE', ascending=False).to_string(index=False))


In [ ]:
# -- Cell 11: Export All 7 Deliverables & Native Models --------------------
import datetime
NM_DIR = os.path.join(OUT_DIR, 'native_models')
os.makedirs(NM_DIR, exist_ok=True)

# 1. Native Models for Production
results['RUL']['prod']['xgb'].save_model(os.path.join(NM_DIR, 'rul_xgb.json'))
results['RUL']['prod']['lgb'].booster_.save_model(os.path.join(NM_DIR, 'rul_lgb.txt'))
results['RUL']['prod']['cat'].save_model(os.path.join(NM_DIR, 'rul_cat.cbm'), format='cbm')
joblib.dump(results['RUL']['prod']['rf'], os.path.join(NM_DIR, 'rul_rf.joblib'))
for tgt in HI_TARGETS:
    joblib.dump(results[tgt]['prod'], os.path.join(NM_DIR, f'{tgt}_models.joblib'))
print('1. Native models exported successfully.')

# 2. Ridge Meta-Model
meta_info = {
    'base_models_order': ALGOS,
    'rul_meta_coefficients': [float(c) for c in results['RUL']['meta'].coef_],
    'rul_meta_intercept': float(results['RUL']['meta'].intercept_),
    'component_metas': {tgt: {
        'coefficients': [float(c) for c in results[tgt]['meta'].coef_],
        'intercept': float(results[tgt]['meta'].intercept_)
    } for tgt in HI_TARGETS}
}
with open(os.path.join(OUT_DIR, 'ridge_meta_model.json'), 'w') as f:
    json.dump(meta_info, f, indent=2)
joblib.dump(results['RUL']['meta'], os.path.join(OUT_DIR, 'ridge_rul_meta.joblib'))
print('2. Ridge meta-model exported.')

# 3. Z-Score Scaler
scaler_data = {col: {'mean': float(ref.loc['mean', col]), 'std': float(ref.loc['std', col])}
               for col in SENSOR_COLS if col in ref.columns}
with open(os.path.join(OUT_DIR, 'zscore_scaler.json'), 'w') as f:
    json.dump(scaler_data, f, indent=2)
pd.DataFrame([{'feature': k, 'mean': v['mean'], 'std': v['std']} for k, v in scaler_data.items()]).to_csv(
    os.path.join(OUT_DIR, 'zscore_scaler.csv'), index=False)
print('3. Z-score scaler exported.')

# 4. Feature Order Specification
feat_info = {
    'total_features': len(RUL_FEATS),
    'feature_order': RUL_FEATS,
    'cycle_leakage': 'EXCLUDED (Model relies strictly on physical wear and stress)',
    'component_targets': HI_TARGETS,
    'rul_target': 'RUL'
}
with open(os.path.join(OUT_DIR, 'features_and_preprocessing.json'), 'w') as f:
    json.dump(feat_info, f, indent=2)
print('4. Features configuration exported.')

# 5. Label Definitions Document
label_md = r"""# RUDRA N-CMAPSS Label Definitions (Peer-Reviewed Spec)
## 1. Physical Target Definitions
- **HI_turbine**: Derived from NASA T_dev HPT efficiency and flow modifiers. Normalized from 0.0 (healthy baseline) to 1.0 (failure threshold).
- **HI_fan & HI_compressor**: In N-CMAPSS DS01, NASA injected no synthetic fault into Fan or LPC/HPC. These are evaluated as nominal baseline.
- **RUL Target**: Operational flight cycles to failure derived directly from NASA Y_dev.
"""
with open(os.path.join(OUT_DIR, 'LABEL_DEFINITIONS.md'), 'w') as f: f.write(label_md)
print('5. LABEL_DEFINITIONS.md exported.')

# 6. Test Verification CSV (350 rows)
verify_rows = test_feat.copy().head(350)
for tgt in HI_TARGETS:
    verify_rows[f'pred_{tgt}'] = np.clip(results[tgt]['test_preds'][:len(verify_rows)], 0.0, 1.0)
    verify_rows[f'gauge_{tgt.replace("HI_","")}_pct'] = np.round((1.0 - verify_rows[f'pred_{tgt}']) * 100.0, 1)
verify_rows['pred_RUL_raw'] = np.round(raw_test_rul[:len(verify_rows)], 2)
verify_rows['final_kalman_RUL'] = np.round(kalman_test_rul[:len(verify_rows)], 2)
export_cols = ['unit','cycle'] + SENSOR_COLS[:4] + \
              [f'pred_{t}' for t in HI_TARGETS] + \
              [f'gauge_{t.replace("HI_","")}_pct' for t in HI_TARGETS] + \
              ['RUL', 'pred_RUL_raw', 'final_kalman_RUL']
verify_rows[export_cols].to_csv(os.path.join(OUT_DIR, 'test_verification_samples.csv'), index=False)
print('6. test_verification_samples.csv exported.')

# 7. Metadata and Metrics JSON
meta_json = {
    'architecture': 'RUDRA N-CMAPSS Production Prognostics',
    'dataset': 'NASA N-CMAPSS DS01-005',
    'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'system_rul_raw_r2': round(raw_r2_all, 4),
    'system_rul_raw_rmse': round(raw_rmse_all, 2),
    'system_rul_kalman_r2': round(kalman_r2_all, 4),
    'system_rul_kalman_rmse': round(kalman_rmse_all, 2),
    'component_metrics': comp_metrics,
    'train_engines': [int(u) for u in train_units],
    'test_engines': [int(u) for u in test_units]
}
with open(os.path.join(OUT_DIR, 'metadata_and_metrics.json'), 'w') as f:
    json.dump(meta_json, f, indent=2)
print('7. metadata_and_metrics.json exported.')


In [ ]:
# -- Cell 12: Component Trajectories & Adaptive Kalman Trajectory -----------
sample_unit = test_df['unit'].unique()[0]
unit_data   = test_feat[test_feat['unit'] == sample_unit].sort_values('cycle')
unit_idx    = np.where(test_feat['unit'].values == sample_unit)[0]

fig, axes = plt.subplots(len(HI_TARGETS) + 1, 1, figsize=(12, 3.2*(len(HI_TARGETS)+1)), sharex=True)
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, tgt in enumerate(HI_TARGETS):
    gt_hi   = unit_data[tgt].values
    pred_hi = np.clip(results[tgt]['test_preds'][unit_idx], 0.0, 1.0)
    axes[i].fill_between(unit_data['cycle'], gt_hi, alpha=0.2, color=colors[i], label='NASA GT')
    axes[i].plot(unit_data['cycle'], gt_hi,   color=colors[i], lw=2.0, label='Ground Truth (T_dev)')
    axes[i].plot(unit_data['cycle'], pred_hi, color='black',   lw=1.8, ls='--', label='RUDRA Prediction')
    axes[i].set_ylabel('Health Index (HI)', fontsize=9)
    axes[i].set_title(f'{tgt.replace("HI_","").capitalize()} Health Degradation', fontsize=11, fontweight='bold')
    axes[i].legend(loc='upper left', fontsize=9)
    axes[i].set_ylim(-0.05, 1.1); axes[i].grid(True, alpha=0.3)

gt_rul     = unit_data['RUL'].values
raw_rul    = raw_test_rul[unit_idx]
kalman_rul = kalman_test_rul[unit_idx]

axes[-1].plot(unit_data['cycle'], gt_rul,     'k-',  lw=2.5, label='True NASA RUL')
axes[-1].plot(unit_data['cycle'], raw_rul,    'r:',  lw=1.2, alpha=0.6, label='Raw ML Stack (No Cycle Leakage)')
axes[-1].plot(unit_data['cycle'], kalman_rul, 'b--', lw=2.2, label='RUDRA Adaptive Kalman Filter')
axes[-1].set_xlabel('Flight Cycle', fontsize=10)
axes[-1].set_ylabel('RUL (cycles)', fontsize=10)
axes[-1].set_title('Remaining Useful Life Trajectory', fontsize=11, fontweight='bold')
axes[-1].legend(loc='upper right', fontsize=9); axes[-1].grid(True, alpha=0.3)

plt.suptitle(f'RUDRA Component Prognostics - Test Engine Unit {sample_unit}', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'component_degradation_plot.png'), dpi=150)
plt.show()


In [ ]:
# -- Cell 13: Bulletproof Package & Download All Deliverables --------------
import os, sys, shutil
from google.colab import files

OUT_DIR = '/content/rudra_ncmapss_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

zip_base = '/content/RUDRA_NCMAPSS_Artifacts'
shutil.make_archive(zip_base, 'zip', OUT_DIR)
print(f'Archive created: {zip_base}.zip\n')

print('Deliverable Package Contents:')
for root, dirs, fls in os.walk(OUT_DIR):
    for fl in fls:
        full = os.path.join(root, fl)
        sz_mb = os.path.getsize(full) / 1e6
        print(f'  {fl:45s}  {sz_mb:6.2f} MB')

print('\nDownloading ZIP archive to your computer...')
files.download(f'{zip_base}.zip')
